# VOX-SYNAPSE — Stage 2 Interleaved KV Decode

Один Qwen backbone, отдельные Planner/Speaker KV-cache и token-step scheduler. Выбери GPU runtime; A100 предпочтительна для первого измерения.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}
TOOL_LATENCY_MS = 3000  # @param {type:"integer"}

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL")

In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/vox")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml]"], check=True)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

In [ ]:
artifacts = repo_dir / "artifacts" / "colab-stage2"
artifacts.mkdir(parents=True, exist_ok=True)
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
(artifacts / "tests.log").write_text(tests.stdout, encoding="utf-8")
print(tests.stdout)
if tests.returncode != 0:
    raise RuntimeError("Tests failed")

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
command = [
    sys.executable, "-m", "vox.experiments.colab_stage2",
    "--allow-download",
    "--model", MODEL_ID,
    "--tool-latency-ms", str(TOOL_LATENCY_MS),
    "--output-dir", str(artifacts),
]
run = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(artifacts / "model_run.log").write_text(run.stdout, encoding="utf-8")
print(run.stdout)
print("Exit code:", run.returncode)

In [ ]:
import json

report_path = artifacts / "report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Status:", report.get("status"))
    print("Result:", report.get("result_text", ""))
    print("Proof:", json.dumps(report.get("proof", {}), indent=2))
    print("Error:", report.get("error", ""))

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/vox-colab-stage2-logs", "zip", root_dir=artifacts)
print(archive)
files.download(archive)